<a href="https://colab.research.google.com/github/nastaran-farhadi/LLM-BBAC/blob/main/Behavior_base_access_control_in_LLM_in_dynamic_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Step 1:** Generate Historical Data.
Simulate a dataset that contains historical user behavior information.
For example, each record stores the employee’s role, department, access timestamp, number of MFA rejections (which we use as a signal for anomaly), and feedback from the last access. We also assign a “behavior score” to indicate risk.

In [ ]:
import pandas as pd
import random
import datetime

In [ ]:
def generate_historical_dataset(n=5):
    """
    Generate simulated historical behavior data for n users.
    Each record includes user context and access details.
    """
    roles = ["Admin", "Manager", "Employee", "Guest"]
    departments = ["IT", "HR", "Finance", "Sales", "Engineering"]
    ip_addresses = ["192.168.1.10 (Private)", "203.45.67.89 (Public)",
                    "154.21.33.45 (Public)", "10.0.0.5 (Private)", "182.67.90.11 (Public)"]
    geo_locations = ["New York", "London", "Paris", "Singapore", "Berlin"]

    data = []
    for user_id in range(1, n + 1):
        role = random.choice(roles)
        department = random.choice(departments)
        # Simulate a past access time (within the last 30 days)
        access_timestamp = datetime.datetime.now() - datetime.timedelta(days=random.randint(1, 30),
                                                                          hours=random.randint(0, 23))
        # For simplicity, we simulate if the access was accepted ("Yes") or rejected ("No")
        access_outcome = random.choice(["Yes", "No"])
        # MFA failure count: if user fails MFA more times, it is more anomalous.
        mfa_failures = random.randint(0, 3)
        ip_address = random.choice(ip_addresses)
        geo_location = random.choice(geo_locations)
        # Feedback from the last access (could be "Normal" or indicate issues)
        feedback = random.choice(["Normal", "Unusual access time", "Multiple MFA failures", "Suspicious activity"])
        # A simple behavior score: higher value means more risk. (For example, we set it randomly based on risk conditions.)
        if access_outcome == "No" or mfa_failures >= 2 or "Public" in ip_address:
            behavior_score = round(random.uniform(0.6, 1.0), 2)  # High risk
        else:
            behavior_score = round(random.uniform(0.1, 0.5), 2)  # Low risk

        data.append([
            user_id, role, department, access_timestamp, access_outcome,
            mfa_failures, feedback, ip_address, geo_location, behavior_score
        ])

    columns = ["Employee_ID", "Role", "Department", "Access_Timestamp", "Access_Outcome",
               "MFA_Failure_Count", "Feedback_From_Last_Access", "IP_Address & Network_Info",
               "Geo_Location", "Behavior_Score"]

    return pd.DataFrame(data, columns=columns)

# Generate dataset for 20 users
historical_df = generate_historical_dataset(n=20)
#print("Historical Data (first 5 rows):")
#print(historical_df.head())
print("\nHistorical Data")
print(historical_df.to_string(index=False))


Historical Data
 Employee_ID     Role  Department           Access_Timestamp Access_Outcome  MFA_Failure_Count Feedback_From_Last_Access IP_Address & Network_Info Geo_Location  Behavior_Score
           1    Guest          IT 2025-02-27 06:44:25.149143             No                  0     Multiple MFA failures    192.168.1.10 (Private)     New York            0.65
           2 Employee       Sales 2025-03-09 11:44:25.149174             No                  1                    Normal    192.168.1.10 (Private)       Berlin            0.82
           3 Employee     Finance 2025-02-22 08:44:25.149183             No                  0       Suspicious activity     154.21.33.45 (Public)       London            0.76
           4    Admin          HR 2025-02-13 21:44:25.149193            Yes                  1                    Normal    192.168.1.10 (Private)     New York            0.31
           5  Manager Engineering 2025-02-09 08:44:25.149200            Yes                  2       Un

**Step 2:** Convert Historical Data to Embeddings:
To use the LLM and perform similarity searches, we need to convert each historical record into a text summary and then create an embedding (a numerical vector) from that text. We use the SentenceTransformer model to do this. Then, we store all the embeddings in FAISS (a vector database) for efficient similarity search.

In [ ]:
!pip install -q sentence_transformers
from sentence_transformers import SentenceTransformer
import torch

In [ ]:
!pip install -q faiss-cpu
import faiss

In [ ]:
import numpy as np
import faiss

# Load a small embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device='cpu')

def convert_to_text(row):
    """
    Create a text summary from each historical record.
    """
    return (f"Employee {row['Employee_ID']} ({row['Role']} in {row['Department']}) accessed the system on "
            f"{row['Access_Timestamp']:%Y-%m-%d %H:%M:%S}. Outcome: {row['Access_Outcome']}, MFA Failures: "
            f"{row['MFA_Failure_Count']}, Feedback: {row['Feedback_From_Last_Access']}, using IP: "
            f"{row['IP_Address & Network_Info']} from {row['Geo_Location']}. Behavior Score: {row['Behavior_Score']}.")

# Create text summaries for all historical records
historical_df["Text_Summary"] = historical_df.apply(convert_to_text, axis=1)

# Generate embeddings for these summaries
historical_embeddings = embedding_model.encode(historical_df["Text_Summary"].tolist())

# Build a FAISS index to store these embeddings for similarity search.
dimension = historical_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(np.array(historical_embeddings).astype("float32"))

print("\nFAISS index created with historical embeddings.")



FAISS index created with historical embeddings.


**Step 3:** Process a Real-Time Access Request:
When a new user requests access, we capture the request details (employee ID, role, department, timestamp, MFA failures, IP, location, etc.), convert this information into a text summary, and generate an embedding. Then we use the FAISS index to retrieve the most similar historical records, which provide context for typical behavior.

In [ ]:
def process_realtime_request(realtime_request):
    """
    Convert a real-time access request into a text summary and generate its embedding.
    Then, retrieve similar historical records using FAISS.
    """
    # Create text summary from the real-time request.
    text_summary = (f"Employee {realtime_request['Employee_ID']} ({realtime_request['Role']} in {realtime_request['Department']}) "
                    f"requests access from {realtime_request['Geo_Location']} using {realtime_request['IP_Address & Network_Info']} with "
                    f"{realtime_request['MFA_Failure_Count']} MFA failures.")

    # Generate embedding for the real-time request
    request_embedding = embedding_model.encode([text_summary])

    # Retrieve top 3 similar historical records
    k = 1
    _, indices = faiss_index.search(np.array(request_embedding).astype("float32"), k)
    similar_records = historical_df.iloc[indices[0]]

    return text_summary, similar_records

# Example real-time access request
realtime_request = {
    "Employee_ID": 10,
    "Role": "IT",
    "Department": "IT",
    "Geo_Location": "Berlin",
    "IP_Address & Network_Info": "Public",
    "MFA_Failure_Count": 2
}

request_summary, retrieved_records = process_realtime_request(realtime_request)
print("\nReal-Time Request Summary:")
print(request_summary)
print("\nRetrieved Similar Historical Records:")
print(retrieved_records[["Employee_ID", "Access_Timestamp", "MFA_Failure_Count", "Behavior_Score"]])



Real-Time Request Summary:
Employee 10 (IT in IT) requests access from Berlin using Public with 2 MFA failures.

Retrieved Similar Historical Records:
   Employee_ID           Access_Timestamp  MFA_Failure_Count  Behavior_Score
1            2 2025-03-09 11:44:25.149174                  1            0.82


**Step 4: **Leverage the LLM for Anomaly Detection:
Now we combine the real-time access request details with the retrieved historical context and send this to an LLM. The LLM then analyzes whether the new request is anomalous compared to historical behavior. It will provide an anomaly score (e.g., 0 = normal, 1 = highly anomalous), insights into the differences, and a recommendation (e.g., require additional authentication).

Code Example: Note: In this example, we use a lightweight model via Hugging Face’s pipeline (e.g., a smaller model like "facebook/blenderbot-3B") to avoid crashing Colab. You can replace it with an OpenAI API call or another LLM if desired.

In [ ]:
from transformers import pipeline, AutoTokenizer

# Use a lighter model to reduce memory and token issues.
# Here we use "distilgpt2" as an example.
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
llm_model = pipeline("text-generation", model="distilgpt2", tokenizer=tokenizer, device="cpu")


def detect_anomaly_with_llm(realtime_request, retrieved_records, request_summary, llm_model):
    """
    Constructs a prompt combining the real-time request details and historical context,
    then uses the LLM to generate an analysis including an anomaly score, insights, and recommendations.
    """
    # Format and possibly shorten the historical records text.
    historical_text = "Historical Access Records:\n" + retrieved_records.to_string(index=False, max_colwidth=50)[:200]

    prompt = f"""
You are an access control security expert.
Analyze the following real-time access request compared to historical user behavior.

Real-Time Request:
{request_summary}

{historical_text}

Questions:
1. How anomalous is this access request? Assign an anomaly score between 0.0 (normal) and 1.0 (highly anomalous).
2. Provide insights on any deviations (e.g., unusual access time, unexpected network source, high MFA failures).
3. Recommend an action (e.g., grant access, require additional authentication, flag for review).

Please format your response as:
Anomaly Score: <score>
Insights: <your insights>
Recommendation: <your recommendation>
"""
    # Debug: Print prompt token count to see if truncation might be needed.
    encoded = tokenizer(prompt, return_tensors="pt", truncation=True)
    print("Prompt token count:", encoded.input_ids.shape[1])

    # Generate response from the LLM using max_new_tokens.
    generated = llm_model(prompt, max_new_tokens=50, truncation=True)

    # Check if output is generated
    if not generated or len(generated) == 0:
        raise ValueError("No output was generated by the LLM. Consider revising the prompt or increasing max_new_tokens.")

    response_text = generated[0].get('generated_text', '')
    return response_text

# ----- Example Usage of Step 4 -----
# (Assuming realtime_request, retrieved_records, and request_summary were produced in Step 3)
# For example, from previous steps:
# realtime_request = {
#     "Employee_ID": 10,
#     "Role": "IT",
#     "Department": "IT",
#     "Geo_Location": "Berlin",
#     "IP_Address & Network_Info": "Public",
#     "MFA_Failure_Count": 2
# }
#
# And request_summary and retrieved_records have been computed accordingly.

llm_response = detect_anomaly_with_llm(realtime_request, retrieved_records, request_summary, llm_model)
print("\nLLM Analysis:")
print(llm_response)


In [ ]:
from transformers import pipeline, AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/blenderbot-400M-distill")

# Load the LLM (text generation model)
llm_model = pipeline("text-generation", model="facebook/blenderbot-400M-distill", tokenizer=tokenizer, device="cpu")


Device set to use cpu


In [ ]:
def detect_anomaly_with_llm(realtime_request, retrieved_records, request_summary, llm_model):
    """
    Detects anomalies using an LLM by comparing the current request with historical data.
    """
    # Convert historical records into a limited-length text summary
    historical_text = "Historical Access Records:\n" + retrieved_records.to_string(index=False, max_colwidth=50)[:200]

    # Construct a clear prompt for the LLM
    prompt = f"""
    You are an AI trained in access control security.
    Given a real-time request and historical user behavior, analyze the anomaly risk.

    Real-Time Request:
    {request_summary}

    Historical Records:
    {historical_text}

    Your task:
    - Assign an anomaly score (between 0.0 = normal and 1.0 = highly anomalous).
    - Identify key anomalies (e.g., unusual location, high MFA failures).
    - Recommend an action.

    Please respond in JSON format:
    {{
      "Anomaly_Score": <numeric_value>,
      "Insights": "<brief explanation>",
      "Recommendation": "<action>"
    }}
    """
#Truncate the prompt to 500 characters**
    prompt = prompt[:500]
    # Check token length
    token_count = len(tokenizer(prompt)['input_ids'])
    print("Token count:", token_count)
    if token_count > 512:
        print("Warning: Prompt is too long. Consider reducing historical records.")

    # Get response from LLM
    generated = llm_model(prompt, max_new_tokens=50, truncation=True)

    if not generated or len(generated) == 0:
        return "LLM did not generate a valid response."

    response_text = generated[0].get('generated_text', '')

    return response_text

# Run the function
llm_response = detect_anomaly_with_llm(realtime_request, retrieved_records, request_summary, llm_model)

print("\nLLM Analysis:")
print(llm_response)


Token count: 212


IndexError: index out of range in self

In [ ]:
!pip install openai


In [ ]:
import openai
import json

# Set up OpenAI API key
openai_client = openai.OpenAI(api_key="")  # Replace with your actual key

def detect_anomaly_with_gpt(realtime_request, retrieved_records, request_summary):
    """
    Uses OpenAI's GPT model to analyze access control anomalies.
    """

    if retrieved_records.empty:
        return "Error: No historical records found for comparison."

    # Convert historical records into a limited-length text summary
    historical_text = "Historical Access Records:\n" + retrieved_records.to_string(index=False, max_colwidth=50)[:500]

    # Construct a prompt for GPT
    prompt = f"""
    You are an AI trained in access control security.
    Given a real-time access request and historical user behavior, analyze the anomaly risk.

    Real-Time Request:
    {request_summary}

    Historical Records:
    {historical_text}

    Your task:
    - Assign an anomaly score (between 0.0 = normal and 1.0 = highly anomalous).
    - Identify key anomalies (e.g., unusual location, high MFA failures).
    - Recommend an action.

    Please respond in JSON format:
    {{
      "Anomaly_Score": <numeric_value>,
      "Insights": "<brief explanation>",
      "Recommendation": "<action>"
    }}
    """

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4",  # Use "gpt-3.5-turbo" if GPT-4 is not available
            messages=[
                {"role": "system", "content": "You are an expert in access control and cybersecurity."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=200
        )

        # Extract response content
        response_text = response.choices[0].message.content

        # Try parsing as JSON (if GPT outputs a structured response)
        try:
            parsed_response = json.loads(response_text)
        except json.JSONDecodeError:
            parsed_response = {"Anomaly_Score": "N/A", "Insights": response_text, "Recommendation": "Review manually"}

        return parsed_response

    except Exception as e:
        return f"Error: {str(e)}"

# Example usage
llm_response = detect_anomaly_with_gpt(realtime_request, retrieved_records, request_summary)
print("\nLLM Analysis:")
print(llm_response)



LLM Analysis:
{'Anomaly_Score': 0.8, 'Insights': 'Employee 10 in IT is requesting access from a public network, coupled with two MFA failure attempts which is quite unusual. Furthermore, the access request from Berlin might be anomalous considering the historical data which does not show access requests from this geo location.', 'Recommendation': 'Deny access request and send an immediate alert to the security team for intervention as well as notify the user to verify recent activities.'}


In [ ]:
# Install
!pip install code2flow

# Generate flowchart
!code2flow src/ --output flowchart.png


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for code2flow: filename=code2flow-2.5.1-py3-none-any.whl size=439527 sha256=66ebefc71f4d32f6249882a7a5798eff9627ef81bd0e695b9308c73d9ba49f2b
  Stored in directory: /root/.cache/pip/wheels/c6/bb/7a/53e69c369cc88f7117776cf4bbd693f9294c53bdb09cf600ee
Successfully built code2flow
Traceback (most recent call last):
  File "/usr/local/bin/code2flow", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/code2flow/engine.py", line 860, in main
    code2flow(
  File "/usr/local/lib/python3.11/dist-packages/code2flow/engine.py", line 713, in code2flow
    sources, language = get_sources_and_language(raw_source_paths, language)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/code2flow/engine.py", line 306, in get_sources_and_lang